## ex04_ab-test.ipynb

In [1]:
import pandas as pd
import sqlite3

## Use the sqlite3 library to create a connection to the database.

In [2]:
db_path = "../data/checking-logs.sqlite"
connection = sqlite3.connect(db_path)

## Using one query per group, create two dataframes, test_results and control_results, with the columns "time" and "avg_diff" and two rows

In [3]:
query = """
        WITH diffs AS (
            SELECT
                uid,
                CASE
                    WHEN unixepoch(first_commit_ts) < unixepoch(first_view_ts) THEN 'before'
                    ELSE 'after'
                END AS time,
                (unixepoch(first_commit_ts) - deadlines.deadlines) / 3600.0 AS diff
            FROM test
            JOIN deadlines ON deadlines.labs = test.labname
            WHERE labname <> 'project1'
        )
        SELECT
            time,
            AVG(diff) AS avg_diff
        FROM diffs
        WHERE uid IN (
            SELECT uid
            FROM diffs
            GROUP BY uid
            HAVING COUNT(DISTINCT time) = 2
        )
        GROUP BY time
        ORDER BY time
        """
test_results = pd.read_sql(query, connection)
display(test_results)

,time,avg_diff
0,after,-105.229241
1,before,-61.156632


## We are still not using the lab project1.

In [4]:
query = """
        SELECT
            CASE
                WHEN julianday(c.first_commit_ts) < (SELECT AVG(julianday(first_view_ts)) FROM test) THEN 'before'
                ELSE 'after'
            END AS time,
            ROUND(AVG((julianday(c.first_commit_ts) - julianday(DATETIME(d.deadlines, 'unixepoch'))) * 24), 6) AS avg_diff
        FROM control c
        JOIN deadlines d ON c.labname = d.labs
        WHERE c.labname != 'project1'
        GROUP BY time
        ORDER BY time;
        """
control_results = pd.read_sql(query, connection)
display(control_results)

,time,avg_diff
0,after,-113.232196
1,before,-99.901295


## Have the answer ready: "Did the hypothesis turn out to be true, and does the page affect students' behavior?"

In [5]:
test_results.columns = ['time', 'avg_diff']
control_results.columns = ['time', 'avg_diff']

test_before = test_results[test_results.time=='before'].avg_diff.values[0]
test_after = test_results[test_results.time=='after'].avg_diff.values[0]
control_before = control_results[control_results.time=='before'].avg_diff.values[0]
control_after = control_results[control_results.time=='after'].avg_diff.values[0]

test_effect = test_after - test_before
control_effect = control_after - control_before

print(f"Test = {test_effect:.2f}, Control = {control_effect:.2f}, Difference = {test_effect - control_effect:.2f}")
print("Hypothesis true: page affect students' behavior")

Test = -44.07, Control = -13.33, Difference = -30.74
Hypothesis true: page affect students' behavior


## Close the connection.

In [6]:
connection.close()